# 00 Prepare External Phonons

This notebook prepares the bundled Graphene phonon fixtures and reproduces the Fig. 1 role from the DFTBephy paper: electronic bands and phonon dispersion. The electronic model is created directly from bundled matsci-0-3 SKF files through DeePTB `DFTBSK`; no `.pth` checkpoint or machine-local dftbephy checkout is required.

In [ ]:
from pathlib import Path
import json
import numpy as np
from dptb.nn.dftbsk import DFTBSK
from dptb.postprocess.unified import TBSystem
from dptb.postprocess.unified.eph import Phonons

ROOT = Path.cwd()
DATA = ROOT / "data"
WORK = ROOT / "work"
WORK.mkdir(exist_ok=True)

def regularize_tiny_negative_frequencies(phonons, tol=1e-3):
    frequencies = np.array(phonons.frequencies, copy=True)
    min_frequency = float(np.min(frequencies))
    if min_frequency < -tol:
        raise ValueError(
            f"Found phonon frequency {min_frequency} THz below tolerance {-tol} THz; "
            "this looks like a real imaginary mode, not acoustic numerical noise."
        )
    clipped = int(np.count_nonzero(frequencies < 0.0))
    frequencies[frequencies < 0.0] = 0.0
    if clipped:
        phonons = Phonons(
            qpoints=phonons.qpoints,
            frequencies=frequencies,
            eigenvectors=phonons.eigenvectors,
            masses=phonons.masses,
            cell=phonons.cell,
            scaled_positions=phonons.scaled_positions,
            metadata={
                **phonons.metadata,
                "negative_frequency_regularization": "clipped_to_zero",
                "negative_frequency_tolerance_thz": tol,
                "negative_frequency_min_before_clip_thz": min_frequency,
                "negative_frequency_clipped_count": clipped,
            },
        )
    return phonons

SK_DATA = DATA / "skf" / "matsci-0-3"
STRUCTURE = ROOT / "graphene.vasp"
PHONOPY_YAML = DATA / "graphene" / "phonons" / "phonopy_disp.yaml"
FORCE_SETS = DATA / "graphene" / "phonons" / "FORCE_SETS"

for path, message in [
    (SK_DATA / "C-C.skf", "The bundled matsci-0-3 C-C.skf file is required."),
    (STRUCTURE, "The bundled graphene.vasp structure is required."),
    (PHONOPY_YAML, "The bundled phonopy_disp.yaml file is required."),
    (FORCE_SETS, "The bundled FORCE_SETS file is required."),
]:
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. {message}")

BASIS = {"C": ["2s", "2p"]}

model = DFTBSK(
    basis=BASIS,
    skdata=str(SK_DATA),
    overlap=True,
    dtype="float64",
    device="cpu",
    interp_method="smooth_intp",
    r_max={'C': 6.349479778742587},
)
model.eval()

system = TBSystem(data=str(STRUCTURE), calculator=model)
print(system.atoms)
print(system.model.name, system.model.basis)


## View Graphene structure

Use ASE's notebook viewer for the bundled primitive graphene structure. The `ngl` viewer is interactive when `nglview` is installed; otherwise the cell falls back to ASE's inline `x3d` viewer.


In [ ]:
from ase.visualize import view

try:
    view(system.atoms, viewer="ngl")
except Exception:
    view(system.atoms, viewer="x3d")

## Select q-points and electronic k-points

The q-points below are intentionally small for an example notebook. For production, replace them with a converged path or mesh.

In [ ]:
q_path = np.array([
    [0.0, 0.0, 0.0],
    [-0.5, -0.5, 0.0],
    [-2.0 / 3.0, -1.0 / 3.0, 0.0],
    [0.0, 0.0, 0.0],
], dtype=float)

q_mesh = np.array([
    [0.0, 0.0, 0.0],
    [0.5, 0.0, 0.0],
    [0.0, 0.5, 0.0],
    [0.5, 0.5, 0.0],
], dtype=float)

fixed_kpoints = np.array([[0.0, 0.0, 0.0]], dtype=float)
np.save(WORK / "q_path.npy", q_path)
np.save(WORK / "q_mesh.npy", q_mesh)
(WORK / "fixed_kpoints.json").write_text(json.dumps({"kpoints": fixed_kpoints.tolist()}, indent=2))
print(WORK)


## Convert phonopy output

`Phonons.from_phonopy_file(...)` calls phonopy as a parser. DeePTB still does not calculate phonons or force constants.

In [ ]:
phonons_path = Phonons.from_phonopy_file(
    PHONOPY_YAML,
    qpoints=q_path,
    force_sets_filename=str(FORCE_SETS),
)

phonons_path = regularize_tiny_negative_frequencies(phonons_path)
phonons_path.save_npz(WORK / "phonons_qpath.npz")

phonons_mesh = Phonons.from_phonopy_file(
    PHONOPY_YAML,
    qpoints=q_mesh,
    force_sets_filename=str(FORCE_SETS),
)

phonons_mesh = regularize_tiny_negative_frequencies(phonons_mesh)
phonons_mesh.save_npz(WORK / "phonons_mesh.npz")

print("q-path phonons:", phonons_path.qpoints.shape, phonons_path.frequencies.shape)
print("mesh phonons:", phonons_mesh.qpoints.shape, phonons_mesh.frequencies.shape)
print("metadata:", phonons_mesh.metadata)

## Quick electronic sanity check

In [ ]:
kpoints = np.array([[0.0, 0.0, 0.0], [1/3, 1/3, 0.0]], dtype=float)
_, evals = system.get_eigenvalues(k_points=kpoints)
evals = np.asarray(evals.detach().cpu() if hasattr(evals, "detach") else evals)

print(evals.shape)
print(evals[:, :8])

## Figure: electronic bands and phonon q-path

The band plot mirrors the role of Fig. 1b in the DFTBephy paper. The phonon plot uses the same q-path prepared above and mirrors Fig. 1c at the smaller example resolution.

In [ ]:
import matplotlib.pyplot as plt

def interpolate_segment(points, n_per_segment=40):
    kpts = []
    x = []
    ticks = [0.0]
    distance = 0.0
    for start, stop in zip(points[:-1], points[1:]):
        segment = np.linspace(start, stop, n_per_segment, endpoint=False)
        for point in segment:
            if kpts:
                distance += float(np.linalg.norm(point - kpts[-1]))
            kpts.append(point)
            x.append(distance)
        ticks.append(distance + float(np.linalg.norm(stop - segment[-1])))
    kpts.append(points[-1])
    if len(kpts) > 1:
        distance += float(np.linalg.norm(kpts[-1] - kpts[-2]))
    x.append(distance)
    ticks[-1] = distance
    return np.asarray(kpts), np.asarray(x), np.asarray(ticks)

hs_points = np.array([
    [0.0, 0.0, 0.0],
    [-0.5, -0.5, 0.0],
    [-2.0 / 3.0, -1.0 / 3.0, 0.0],
    [0.0, 0.0, 0.0],
], dtype=float)
labels = ["G", "M", "K", "G"]
band_kpoints, band_x, band_ticks = interpolate_segment(hs_points, n_per_segment=30)
_, band_evals = system.get_eigenvalues(k_points=band_kpoints)
band_evals = np.asarray(band_evals.detach().cpu() if hasattr(band_evals, "detach") else band_evals)

phonon_plot = regularize_tiny_negative_frequencies(Phonons.from_phonopy_file(
    PHONOPY_YAML,
    qpoints=band_kpoints,
    force_sets_filename=str(FORCE_SETS),
))
phonon_mev = phonon_plot.frequencies * 4.13566553853599

fig, (ax_band, ax_ph) = plt.subplots(1, 2, figsize=(8.5, 3.2), dpi=140)
for iband in range(min(8, band_evals.shape[1])):
    ax_band.plot(band_x, band_evals[:, iband], color="0.2", lw=1.0)
for tick in band_ticks:
    ax_band.axvline(tick, color="0.85", lw=0.8)
ax_band.set_xticks(band_ticks)
ax_band.set_xticklabels(labels)
ax_band.set_ylabel("E [eV]")
ax_band.set_title("Electronic bands")
ax_band.grid(alpha=0.25)

for imode in range(phonon_mev.shape[1]):
    ax_ph.plot(band_x, phonon_mev[:, imode], lw=1.0)
for tick in band_ticks:
    ax_ph.axvline(tick, color="0.85", lw=0.8)
ax_ph.set_xticks(band_ticks)
ax_ph.set_xticklabels(labels)
ax_ph.set_ylabel("hbar omega [meV]")
ax_ph.set_title("Phonon dispersion")
ax_ph.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(WORK / "figure_00_bands_phonons.png", dpi=200)
plt.show()